# Hospital FIAP Assistant — Versão JupyterLite

**Tech Challenge Fase 3 (IA para Devs)** — demo no navegador via [JupyterLite](https://jupyter.org/try-jupyter/lab/).

## Como usar

1. Abra https://jupyter.org/try-jupyter/lab/
2. Faça upload de **dois arquivos** desta pasta `jupyterlite/`:
   - `hospital_fiap_assistant_lite.ipynb` (este notebook)
   - `lite_core.py` (módulo Python com dados e lógica)
3. Execute as células em ordem (`Run` → `Run All Cells`)

> **Limitação:** JupyterLite roda Python no browser (sem GPU, sem `torch`/`LangChain`/`LangGraph`). Esta versão **reproduz os conceitos das aulas** com Python puro. Para treino LoRA real, use o notebook `02_fine_tuning_colab.ipynb` no Google Colab.

## Mapa das aulas (Fase 3)

| Seção | Disciplina / Aula |
|-------|-------------------|
| 1 | Fine-tuning — Preparando dados |
| 2 | Fine-tuning — LoRA (simulado) |
| 3 | RAG para documentos |
| 4 | LangChain — chains, prompts, tools |
| 5 | LangGraph — fluxo clínico automatizado |
| 6 | Segurança — guardrails e logs |

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

try:
    import lite_core as lc
    print("✓ lite_core carregado de", NOTEBOOK_DIR / "lite_core.py")
except ImportError as exc:
    raise ImportError(
        "Faça upload do arquivo lite_core.py na mesma pasta deste notebook."
    ) from exc

## 1. Preparação de dados (Aula: Fine-tuning — Preparando os dados)

Gera pares **instrução / contexto / resposta** a partir dos protocolos sintéticos do hospital — mesmo formato usado no fine-tuning real.

In [ ]:
import pandas as pd

datasets = lc.prepare_dataset_demo()
counts = {split: len(rows) for split, rows in datasets.items()}
print("Contagem por split:", counts)
print("Total:", sum(counts.values()))

rows = []
for split, records in datasets.items():
    for record in records[:2]:
        rows.append({
            "split": split,
            "instruction": record["instruction"][:80] + "...",
            "output": record["output"][:100] + "...",
        })
pd.DataFrame(rows)

In [ ]:
sample = datasets["train"][0]
print("Exemplo de registro formatado para fine-tuning:\n")
print(sample["text"][:600])

## 2. Fine-tuning LoRA — simulação (Aula: Fine-tuning de LLMs)

No browser não há GPU nem `torch`. Simulamos as **épocas, loss e hiperparâmetros LoRA** para demonstrar o pipeline. O treino real está em `fine_tuning/train.py` ou no Colab.

In [ ]:
metrics = lc.simulate_lora_training(epochs=3, steps_per_epoch=10)
print("Modelo base:", metrics["base_model"])
print("Método:", metrics["method"])
print("Épocas:", metrics["epochs"])
print("Loss final:", metrics["final_loss"])
print("Adapter:", metrics["adapter_path"])
print("\nNota:", metrics["note"])

In [ ]:
import matplotlib.pyplot as plt

history = metrics["loss_history"]
steps = [h["step"] for h in history]
losses = [h["loss"] for h in history]

plt.figure(figsize=(8, 4))
plt.plot(steps, losses, marker="o", markersize=3)
plt.title("Curva de loss — LoRA simulado")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.show()

## 3. RAG sobre protocolos (Aula: RAG para documentos)

Busca por palavras-chave nos 15 protocolos sintéticos — equivalente ao fallback do projeto quando ChromaDB não está disponível.

In [ ]:
QUERY = "Protocolo para febre e tosse?"
chunks = lc.buscar_protocolo(QUERY, k=3)

print(f"Query: {QUERY}\n")
for i, chunk in enumerate(chunks, 1):
    print(f"--- Resultado {i}: {chunk['source']} (score={chunk['score']}) ---")
    print(chunk["content"][:400], "...\n")

## 4. Assistente LangChain-style (Aulas: Chains, Prompts, Tools, Agents)

Pipeline: **consultar prontuário (SQLite)** → **RAG** → **prompt clínico** → **Mock LLM** → **guardrails** → **log**.

In [ ]:
print(lc.consultar_prontuario("PAC-001"))

In [ ]:
result = lc.run_assistant("PAC-001", "Protocolo para febre e tosse?")

print("Paciente: PAC-001")
print("Pergunta: Protocolo para febre e tosse?")
print("\nResposta:")
print(result["response"])
print("\nFontes:", result["sources"])
print("Requer validação humana:", result["requires_human_validation"])
print("Flags:", result["flags"])

## 5. Fluxo LangGraph-style (Aulas: StateGraph, RAG, Multi-Agent)

```
triagem → verificar_exames → [alerta | prontuário] → agente_protocolo
         → agente_auditoria → registrar_log → END
```

PAC-002 tem exame pendente → ramo de **alerta**.

In [ ]:
wf = lc.run_workflow("PAC-002", "Qual a conduta atual?")

print("Sugestão:")
print(wf["suggestion"])
print("\nAlertas:")
for alert in wf.get("alerts", []):
    print(" -", alert)
print("\nFontes:", wf.get("sources", []))
print("Fluxo:", " → ".join(wf.get("graph_path", [])))
print("\nRequer validação humana:", wf.get("requires_human_validation"))

In [ ]:
DEMO_CASES = [
    ("PAC-001", "Protocolo para febre e tosse?"),
    ("PAC-002", "Qual a conduta atual?"),
    ("PAC-003", "Conduta de follow-up oncologia?"),
    ("PAC-004", "Há interação medicamentosa?"),
    ("PAC-005", "Consulta geral — próximos passos?"),
]

summary = []
for patient_id, query in DEMO_CASES:
    state = lc.run_workflow(patient_id, query)
    summary.append({
        "paciente": patient_id,
        "exames_pendentes": len(state.get("pending_exams", [])),
        "alertas": len(state.get("alerts", [])),
        "fontes": ", ".join(state.get("sources", [])[:2]),
        "fluxo": " → ".join(state.get("graph_path", [])[-3:]),
    })

pd.DataFrame(summary)

## 6. Guardrails e auditoria (Tech Challenge — Segurança)

Teste de bloqueio de prescrição direta e visualização do log append-only.

In [ ]:
unsafe = "Prescrevo amoxicilina 500mg de 8/8h por 7 dias."
safe_sources = ["protocolo_febre_v2.md"]
check = lc.validate_response(unsafe, safe_sources)
print("Texto original:", unsafe)
print("Válido:", check["valid"])
print("Flags:", check["flags"])
print("Texto sanitizado:", check["sanitized_text"])

In [ ]:
print(f"Total de interações registradas: {len(lc.AUDIT_LOG)}\n")
for entry in lc.AUDIT_LOG[-3:]:
    print(f"[{entry['timestamp'][:19]}] {entry['mode']} — {entry['patient_id']}")
    print(f"  Query: {entry['query'][:60]}...")
    print(f"  Fontes: {entry.get('sources', [])}")
    print()

## 7. Demo interativa

Altere `PATIENT_ID` e `QUERY` abaixo e execute a célula.

In [ ]:
PATIENT_ID = "PAC-001"  # PAC-001 a PAC-005
QUERY = "Protocolo para febre e tosse?"
MODE = "workflow"  # "assistant" ou "workflow"

if MODE == "assistant":
    out = lc.run_assistant(PATIENT_ID, QUERY)
    print(out["response"])
else:
    out = lc.run_workflow(PATIENT_ID, QUERY)
    print(out["suggestion"])
    if out.get("alerts"):
        print("\nAlertas:", out["alerts"])
    print("\nFluxo:", " → ".join(out.get("graph_path", [])))